# Proyecto - Aprendizaje de Máquina

## Librerías

In [ ]:
!python -m pip install -r requirements.txt
!nbdime config-git --enable --global
!nbstripout --install --global

In [ ]:
import os
import pandas as pd
import re
import matplotlib.pyplot as plt
import nltk
nltk.download('punkt_tab')
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('stopwords')
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## Limpieza y transformación de datos

### Carga de datos

In [ ]:
# Carga el archivo CSV
def load_data(filename: str, train: bool = True) -> pd.DataFrame:
    filepath = os.path.join(os.getcwd(), filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        # Guarda el header
        header = f.readline()
        # raw_data almacena los pares de texto y década
        raw_data = [header.strip().split(',')]
        content = f.read() # Archivo con datos completos
        # Extrae los datos a partir de una expresión regular que busca 
        # texto entre comillas dobles seguido de una coma, tres dígitos y un salto de línea
    if train:
        regular_expression = r'"(?:[^"]|"")*",\d{3}\n'
    else:
        regular_expression = r'\d+,"(?:[^"]|"")*"\n'
    joint_data = re.findall(regular_expression, content, flags=re.MULTILINE) #Se guarda en una lista
    for data in joint_data:
        data = data.strip().replace('\n', '') # Elimina saltos de línea y espacios extra
        if train:
            text = data[:-4] # Texto sin la década ni la coma separadora
        else:
            text = data[data.find(',')+1:] # Texto sin el ID ni la coma separadora

        text = text[1:-1].strip() # Elimina comillas externas y espacios extra
        
        if train:
            decade = data[-3:] # Década (últimos 3 caracteres)
            raw_data.append([text, int(decade)])
        else:
            id = data[:data.find(',')] # ID (antes de la coma)
            raw_data.append([int(id), text])
    data = pd.DataFrame(raw_data[1:], columns=raw_data[0])
    return data
data = load_data('data/train.csv')
print("Total de registros cargados:", len(data)) # Muestra el número de registros cargados
print(data.sample(5))


In [ ]:
print("Registros únicos:", data["text"].nunique())
duplicate_rows = data[data.duplicated(subset=['text'], keep=False)]
print(duplicate_rows.sort_values(by='text'))

In [ ]:
data.drop_duplicates(subset=['text'], inplace=True)
print("Total de registros después de eliminar duplicados:", len(data))

In [ ]:
stemmer = SnowballStemmer("spanish")

In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text) # Reemplaza múltiples espacios por uno solo
    text = re.sub(r'(\w+)([-¬>])\s+(\w+)', r'\1\3', text) #Unir palabras con guiones
    tokens = word_tokenize(text.lower())
    for i, t in enumerate(tokens):
        if len(t) > 18:
            tokens[i] = ''
        if re.findall(r'[^a-záéíóúüñ\s\d]', t):
            tokens[i] = ''
    tokens = [t for t in tokens if t]
    text = " ".join([stemmer.stem(t)for t in tokens])
    return text

In [ ]:
data['clean_text'] = data['text'].apply(clean_text)
data['clean_text'].to_csv('data/clean_data.csv', index=False)
print(data['clean_text'].sample(10))

## MultinomialNB

In [ ]:
# Validacion cruzada por modelo (accuracy)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

X_text = data['clean_text']
y = data['decade']
stop_words_es = set(nltk.corpus.stopwords.words('spanish'))
vectorizer = TfidfVectorizer(
    stop_words=list(stop_words_es),
    max_features=50000,
    min_df=3,
    ngram_range=(1, 1)
)
X_tfidf = vectorizer.fit_transform(X_text)

print("Tamaño del vocabulario:", len(vectorizer.get_feature_names_out()))


In [ ]:
# Modelo 3: MultinomialNB
model = MultinomialNB(
    alpha=0.1, 
    fit_prior=False
)

In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)
from sklearn.metrics import classification_report, accuracy_score
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## Datos de evaluación

In [ ]:
eval_data = load_data('data/eval.csv', train=False)
eval_data['clean_text'] = eval_data['text'].apply(clean_text)
X_eval_tfidf = vectorizer.transform(eval_data['clean_text'])
eval_predictions = model.predict(X_eval_tfidf)
eval_data['answer'] = eval_predictions
eval_data[['id', 'answer']].to_csv('data/answers.csv', index=False)